AI BROCHURE GENERATOR - USING OLLAMA

In [1]:
# imports 
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
MODEL = "llama3.2"

load_dotenv()
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} using {MODEL} locally")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
    )

    result = response.choices[0].message.content

    # 🔹 Step 1: Safely parse JSON
    try:
        links_data = json.loads(result)
    except json.JSONDecodeError:
        print("⚠️ Model did not return valid JSON. Raw output:")
        print(result)
        return {"links": []}

    # 🔹 Step 2: Clean and validate links
    clean_links = [
        link.strip()
        for link in links_data.get("links", [])
        if isinstance(link, str) and link.strip().startswith("http")
    ]

    print(f"Found {len(clean_links)} valid links")

    return {"links": clean_links}

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com using llama3.2 locally
Found 0 valid links


{'links': []}

In [13]:
def fetch_page_and_all_relevant_links(url):
    print("Fetching landing page...")
    contents = fetch_website_contents(url)

    print("Selecting relevant links...")
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n\n"

    print("Fetching relevant pages...")

    for link in relevant_links["links"]:
        print(f"Fetching: {link['url']}")
        page_content = fetch_website_contents(link["url"])

        result += f"\n\n## {link['type'].title()}:\n"
        result += page_content

    return result

In [14]:
print(fetch_page_and_all_relevant_links("http://edwarddonner.com"))

Fetching landing page...
Selecting relevant links...
Selecting relevant links for http://edwarddonner.com using llama3.2 locally
⚠️ Model did not return valid JSON. Raw output:
Here are the relevant links extracted from the website http://edwarddonner.com and included in a JSON format:

```
{
    "links": [
        {"type": "About page", "url": "/about-me-and-about-nebula/"},
        {"type": "Company page", "url": "/"},
        {"type": "Blog (Posts)". "url": "/posts/"}
    ]
}
```

Here's my reasoning behind choosing these links for a brochure about the company:

1. **About me and About Nebula**: This link is likely to provide an overview of Edward Donner's background, experience, and interests, which can be relevant to learn more about him as a representative of the company.
2. **Company homepage**: The main URL of the website serves as a straightforward introduction to the company, making it a good choice for a brochure that aims to showcase the company's presence online.

I exclud

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages
from a company website and creates a professional brochure.

The brochure should be suitable for:
- Prospective customers
- Investors
- Potential recruits

Respond in markdown format.
Do not use code blocks.

Include:
- Company Overview
- Products/Services
- Market Position
- Company Culture
- Careers/Jobs (if available)
- Call to Action
"""

In [16]:
def get_brochure_user_prompt(company_name, url):
    combined_content = fetch_page_and_all_relevant_links(url)

    user_prompt = f"""
You are analyzing a company called: {company_name}

Below are the contents of its website.
Use this information to create a professional brochure.

{combined_content}
"""
    return user_prompt[:3000]  

In [17]:
def create_brochure(company_name, url):
    print("Generating brochure using Ollama...")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        temperature=0.7
    )

    brochure = response.choices[0].message.content
    display(Markdown(brochure))

In [18]:
create_brochure("Edwarddonner", "http://edwarddonner.com")

Generating brochure using Ollama...
Fetching landing page...
Selecting relevant links...
Selecting relevant links for http://edwarddonner.com using llama3.2 locally
Found 0 valid links
Fetching relevant pages...


**Edward Donner Brochure**

**Company Overview**
Edward Donner is a cutting-edge AI technology company founded by Ed, a seasoned expert in the field. Our mission is to empower individuals to discover their potential and pursue their reason for being through innovative applications of Artificial Intelligence.

**Products/Services**

* **AI Curriculum**: A comprehensive online course platform that teaches AI engineers and enthusiasts how to build proficient agents.
* **Connect Four**: An arena where LLMs (Large Language Models) compete in a battle of diplomacy and deviousness, providing an engaging way to test and compare AI models.
* **Outsmart**: A resource hub offering tutorials, guides, and tools for building and deploying AI solutions.

**Market Position**
Edward Donner takes its place at the forefront of AI innovation, bridging the gap between research and practical application. Our products and services cater to a diverse range of customers, from aspiring AI engineers to established professionals seeking to stay ahead in their field.

**Company Culture**

At Edward Donner, we value:

* **Collaboration**: We foster a culture of open communication and collective growth.
* **Innovation**: We encourage experimentation, creativity, and calculated risk-taking.
* **Community**: We nurture relationships with our customers, partners, and fellow professionals to create a supportive network.

**Careers/Jobs**
Join us in shaping the future of AI! Our team is currently accepting applications for talented individuals who share our passion for innovation and learning. Check out our job openings and submit your application today!

**Call to Action**
Stay ahead of the curve and unlock your potential with Edward Donner's cutting-edge AI solutions. Subscribe to our newsletter to receive exclusive updates, resources, and invitations to join our community.

**Get in Touch**
Email: [ed@edwarddonner.com](mailto:ed@edwarddonner.com)
Website: www.edwarddonner.com
Follow us on LinkedIn, Twitter, Facebook, and subscribe to our newsletter to stay up-to-date with the latest news and developments from Edward Donner.

In [19]:
create_brochure("HuggingFace", "https://huggingface.co/")

Generating brochure using Ollama...
Fetching landing page...
Selecting relevant links...
Selecting relevant links for https://huggingface.co/ using llama3.2 locally
⚠️ Model did not return valid JSON. Raw output:
Here is the list of relevant links for a brochure about Hugging Face, in JSON format:

{
  "links": [
    {
      "type": "About page",
      "url": "https://huggingface.co/"
    },
    {
      "type": "Enterprise page",
      "url": "/enterprise"
    },
    {
      "type": "Careers/Jobs page",
      "url": "/join"
    },
    {
      "type": "Blog page",
      "url": "https://blog.huggingface.co"
    },
    {
      "type": "News/Updates page",
      "url": "https://status.huggingface.co"
    },
    {
      "type": "GitHub repository",
      "url": "https://github.com/huggingface"
    },
    {
      "type": "Twitter handle",
      "url": "https://twitter.com/huggingface"
    },
    {
      "type": "LinkedIn page",
      "url": "https://www.linkedin.com/company/huggingface/"
   

**HuggingFace: Empowering AI Collaboration**

Welcome to Hugging Face, the ultimate platform for machine learning collaboration. Our mission is to bring together the AI community to create a better future through open-source innovation.

**Company Overview**

At Hugging Face, we're passionate about democratizing access to high-quality pre-trained models, datasets, and applications. Our platform provides a collaborative environment where researchers, developers, and entrepreneurs can work together to build cutting-edge AI solutions. With our expertise in natural language processing (NLP), computer vision, and audio processing, we empower users to accelerate their projects and unlock new possibilities.

**Products/Services**

Our platform offers an extensive range of features and tools:

* **Models**: Browse and download over 2 million pre-trained models, ranging from simple text classification to complex image recognition.
* **Datasets**: Explore over 500,000 public datasets, covering various domains such as NLP, computer vision, and audio processing.
* **Spaces**: Collaborate with others on unlimited public models, datasets, and applications. Our Zero environment provides a fast and efficient way to run your projects.
* **Applications**: Discover 1 million+ applications built using our platform, including ones for text-to-video, image editing, and more.

**Market Position**

Hugging Face is at the forefront of machine learning innovation, with a strong presence in the AI community. We've partnered with leading organizations to provide cutting-edge solutions for various industries, including healthcare, finance, and education. Our expertise has been recognized through numerous awards and publications, solidifying our position as a leader in the field.

**Company Culture**

We value collaboration, openness, and innovation above all else. Our team of experts is passionate about empowering others to achieve their goals through machine learning. We strive to create an inclusive community that welcomes diverse perspectives and encourages continuous growth.

**Careers/Jobs (if available)**

Join our team of talented individuals who share your passion for AI innovation! Check out our [careers page](link) to explore current openings and learn more about our company culture.

**Call to Action**

Ready to unlock the full potential of machine learning? Sign up for Hugging Face today and discover how our platform can accelerate your projects. Join our community, collaborate with others, and start building innovative AI solutions that change the world!

**Get Started**

Browse our [models](link), explore our [datasets](link), or sign up for an account ([link]) to begin your journey on Hugging Face.

Stay ahead of the curve in machine learning innovation. Join us today!